问答链
- 加载向量数据库
- 检索式问答链
- 实验：状态记录

In [1]:
### 加载已持久化的数据库

from langchain_chroma.vectorstores import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

persist_directory = 'data/matplotlib/'

embedding = HuggingFaceEmbeddings()

vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


49


In [3]:
# 测试：提问并搜索
question = "这节课的主要话题是什么"
docs = vectordb.similarity_search(question, k=3)
len(docs)

3

构造检索式问答链

In [2]:
### question 作为 query 用于向量数据库
### 向量数据库提供 k 个相关文档
### 文档和原始提问一起传入LLM以生成最终回答

from langchain_ollama import ChatOllama
from langchain_classic.chains import RetrievalQA

llm = ChatOllama(model='qwen2.5:7b', temperature=0.0)

# 创建检索式回答链
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(),
    chain_type="stuff",
    verbose=True,
)

# 进行检索回答
question = "这节课的主要话题是什么"
result = qa_chain.invoke(input=question)
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
这节课的主要话题是介绍 matplotlib 的基本概念和使用方法，包括 Figure 和 Axes 容器、绘图接口以及样式和颜色的设置。具体内容涉及：

1. **Figure 和 Axes 容器**：解释了 Figure 和 Axes 在 matplotlib 中的角色及其属性。
2. **两种绘图接口**：介绍了显式创建 figure 和 axes 与依赖 pyplot 自动创建的方法。
3. **样式和颜色设置**：讲解了如何使用预定义的样式、自定义样式以及通过 rcparams 设置全局样式，并介绍了五种表示单色颜色的基本方法及 colormap 的多色显示方法。

这些内容为后续深入学习 matplotlib 提供了基础。


文档和原始问题一起输入LLM时，存在上下文长度限制问题：
- MapReduce：长文档切块分别处理(map)，再汇总结果(reduce)
- Refine：按顺序逐块处理并不断在已有的答案上增量更新，滚动记忆规避一次性输入过长
- MapRerank：对每个分块独立生成候选答案并打分，最后选最优结果

基于 prompt template 的检索式问答链

In [3]:
from langchain_core.prompts import PromptTemplate


# build template
template = """
使用一下上下文片段来回答最后的问题。
答案最多使用三个句子。
尽量简明扼要。
如果不知道答案，请回答不知道，不要试图编造。
回答完之后一定要说“感谢提问！”

上下文：```{context}```

问题：```{question}```

回答：
"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

In [4]:
### 基于上述模板构建检索式回答链
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={
        "prompt": QA_CHAIN_PROMPT
    },
)


# 问答
question = "这门课会学习 Python 吗"
result = qa_chain.invoke(input=question)
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
不知道，这个问题与课程内容介绍无关。感谢提问！


In [5]:
# 查看检索到的源文档
print(result["source_documents"][0])

page_content='Figure.images属性：⼀个 FigureImages patch 列表
Figure.lines属性：⼀个 Line2D 实例的列表（很少使⽤）
Figure.legends属性：⼀个 Figure Legend 实例列表（不同于 Axes.legends)
Figure.texts属性：⼀个 Figure Text 实例列表
2. Axes 容器
matplotlib.axes.Axes是 matplotlib 的核⼼。⼤量的⽤于绘图的Artist存放在它内部，并且它有许多辅助⽅法来创
建和添加Artist给它⾃⼰，⽽且它也有许多赋值⽅法来访问和修改这些Artist。
和Figure容器类似，Axes包含了⼀个 patch 属性，对于笛卡尔坐标系⽽⾔，它是⼀个Rectangle；对于极坐标⽽⾔，
它是⼀个Circle。这个 patch 属性决定了绘图区域的形状、背景和边框。
fig = plt.figure()
ax = fig.add_subplot(111)
rect = ax.patch  # axes 的 patch 是一个 Rectangle 实例
rect.set_facecolor('green')
 
Axes有许多⽅法⽤于绘图，如.plot() 、 .text() 、 .hist() 、 .imshow()等⽅法⽤于创建⼤多数常⻅的primitive( 如
Line2D ， Rectangle ， Text ， Image等等）。在primitives中已经涉及，不再赘述。
Subplot 就是⼀个特殊的 Axes ，其实例是位于⽹格中某个区域的 Subplot 实例。其实你也可以在任意区域创建 Axes ，
通过 Figure.add_axes([left,bottom,width,height]) 来创建⼀个任意区域的 Axes ，其中 left,bottom,width,height 都是 [0
—1] 之间的浮点数，他们代表了相对于 Figure 的坐标。
你不应该直接通过Axes.lines和Axes.patches列表来添加图表。因为当创建或添加⼀个对象到图表中时，Axes会做
许多⾃动化的⼯作 :
它会设置 Artist 中 figure 和 axes 的属性，同时默认 Axes 的转换；
它也

基于 MapReduce 的检索式问答链

In [6]:
### map_reduce: 先将每个独立的文档单独发送给llm获取原始答案
###             最后再调用一次llm将这些答案合成最终的答案

qa_chain_mr = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(),
    chain_type="map_reduce",
    verbose=True,
)

question = "这门课会学习 Python 吗"
result = qa_chain_mr.invoke(input=question)
print(result["result"])



> Entering new RetrievalQA chain...


C:\Users\61075\miniforge3\envs\rag\lib\site-packages\langchain_core\language_models\base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))



> Finished chain.
根据提供的信息，这段文字并没有明确提到是否会学习 Python。因此，我不确定这门课程是否包括 Python 的学习内容。


基于Refine的检索式回答链

In [7]:
qa_chain_rf = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(),
    chain_type="refine",
    verbose=True,
)

question = "这门课会学习 Python 吗"
result = qa_chain_rf.invoke(input=question)
print(result["result"])



> Entering new RetrievalQA chain...

> Finished chain.
好的，根据新的上下文信息，我们可以对之前的回答进行调整和补充。以下是针对 `matplotlib` 中样式和颜色使用的详细讲解：

### 一、Matplotlib 的绘图样式（style）

在 `matplotlib` 中，设置绘制样式的最简单方法是在绘制元素时单独设置样式。然而，在做专题报告时，用户通常希望保持整体风格的统一而不必对每张图逐一修改。因此，`matplotlib` 库提供了四种批量修改全局样式的途径：

1. **matplotlib 预先定义样式**：
   `matplotlib` 提供了许多内置的样式供用户使用。这些样式可以通过在脚本中调用 `plt.style.use()` 方法来设置。

   ```python
   import matplotlib.pyplot as plt

   # 使用默认样式绘制图形
   plt.plot([1, 2, 3, 4], [2, 3, 4, 5])
   plt.show()

   # 切换到 ggplot 样式
   plt.style.use('ggplot')
   plt.plot([1, 2, 3, 4], [2, 3, 4, 5])
   plt.show()
   ```

2. **用户自定义 stylesheet**：
   用户可以创建自己的样式文件，并通过 `plt.style.use()` 方法加载这些样式。

   ```python
   # 创建一个简单的样式文件 styles.mplstyle
   with open('styles.mplstyle', 'w') as f:
       f.write("""
       lines.color: red
       lines.linewidth: 2.0
       """)

   plt.style.use('styles.mplstyle')
   plt.plot([1, 2, 3, 4], [2, 3, 4, 5])
   plt.show()
   ```

3. **设置 rcparams**：
   `matplotlib` 的配置可以通过 `rcParams` 字

实验：状态记录

In [13]:
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever()
)

question = "这门课会学习 Python 吗？"
result = qa_chain({"query": question})
print(result["result"])

C:\Users\61075\AppData\Local\Temp\ipykernel_29156\2009365756.py:7: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa_chain({"query": question})


这门课程主要围绕 matplotlib 进行讲解，包括 Figure 和 Axes 的组成、绘图接口以及样式和颜色的使用等。虽然会用到一些 Python 代码来展示如何绘制图表，但这门课的主要内容并不是教授 Python 编程语言本身。因此，我们不能说这门课会系统地学习 Python。不过，理解 Python 基础对于掌握这些可视化技术会有很大帮助。


In [14]:
question = "为什么需要这一前提"
result = qa_chain({"query": question})
result["result"]

### 此时他并不记得前面的提问了，需要引入memory

'您提到的“前提”不够明确，我不确定具体指的是哪个部分。如果您能提供更多的上下文或详细说明一下您的问题，我会尽力给出准确的回答。根据现有的信息，我无法确定您所指的具体内容。'